# ODI chase-success prediction (2023 ICC Men's World Cup)
Walk-through notebook. It imports the project modules in `../src` and reads saved results from `../results`. It was **not executed in Jupyter** by the author (no stored outputs); the code cells were run as a plain script to check they work. Full experiments: `python src/run_experiments.py`.

In [10]:
import sys; sys.path.insert(0,'../src')
import pandas as pd, numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from prep import *
pd.set_option('display.width',200); pd.set_option('display.max_columns',30)

## Phase 1: schema inspection

In [11]:
d, m, p = load_raw()
for name, x in [('deliveries', d), ('matches', m), ('points_table', p)]:
    print(name, x.shape, 'full-row duplicates:', x.duplicated().sum())
    print(pd.DataFrame({'dtype': x.dtypes.astype(str), 'nulls': x.isna().sum()}).T)

deliveries (26119, 22) full-row duplicates: 1
      match_id season start_date venue innings     ball batting_team bowling_team striker non_striker bowler runs_off_bat extras    wides  noballs     byes  legbyes  penalty wicket_type  \
dtype    int64    str        str   str   int64  float64          str          str     str         str    str        int64  int64  float64  float64  float64  float64  float64         str   
nulls        0      0          0     0       0        0            0            0       0           0      0            0      0    25454    26063    26081    25926    26118       25390   

      player_dismissed other_wicket_type other_player_dismissed  
dtype              str               str                    str  
nulls            25390             26118                  26118  
matches (48, 18) full-row duplicates: 0
      season team1 team2 date match_number venue city toss_winner toss_decision player_of_match umpire1 umpire2 reserve_umpire match_referee winner 

## Phases 2-3: cleaning, validation, leakage-safe states

In [12]:
dc, m, p, mt, st, rep = build_all()
print(rep)
print(mt[~mt.included][['chaser','target','i2_runs','winner_runs','exclude_reason']])
print('margin check failures among included matches:', int((~mt[mt.included].margin_ok).sum()))
for k, v in exclusion_counts(st).items(): print(k, v)
df = st[st.eligible].reset_index(drop=True)
df[['match_id','legal_balls','runs','wkts','runs_required','rrr','rate_last12','chase_win']].head()

{'extras_mismatch_rows': 0, 'innings_per_match_ok': True, 'wickets_gt10_innings': 0, 'rows_ball_not_increasing(float truncation)': 6}
      chaser  target  i2_runs  winner_runs                                     exclude_reason
35  Pakistan     402      200         21.0  chase won below first-innings total (DLS-type ...
margin check failures among included matches: 0
all chase deliveries in included matches 11568
terminal (last delivery / target reached / all out / overs done) 47
   last delivery of innings 47
   target reached 24
   all out 21
   overs exhausted 1
impossible (required > 6 x balls left), non-terminal 48
eligible 11473


,match_id,legal_balls,runs,wkts,runs_required,rrr,rate_last12,chase_win
0,1,1,4,0,279,5.598662,24.0,1
1,1,2,4,0,279,5.617450,12.0,1
2,1,3,4,0,279,5.636364,8.0,1
3,1,4,6,0,277,5.614865,9.0,1
4,1,5,10,0,273,5.552542,12.0,1


## Phase 4: association rules (saved results)
Re-mining: see `assoc.py` (`build_transactions`, `apriori`, `fpgrowth`, `outcome_rules`).

In [13]:
from assoc import *
s, attrs = build_transactions(st); T = to_item_lists(s, attrs)
_, ca = apriori(T, 0.05); _, cf = fpgrowth(T, 0.05)
print(len(T), 'transactions;', len(ca), 'frequent itemsets; identical:', ca == cf)
rules = pd.read_csv('../results/association_rules.csv')
rules[['antecedent','consequent','support','confidence','lift','n_matches']].head(10)

1747 transactions; 347 frequent itemsets; identical: True


,antecedent,consequent,support,confidence,lift,n_matches
0,RRR=high(>8) & WKTS_LEFT=moderate(4-6),OUTCOME=Loss,0.113337,1.0,2.077289,19
1,PACE_VS_REQ=behind(<-1) & RRR=high(>8) & WKTS_...,OUTCOME=Loss,0.101889,1.0,2.077289,19
2,RRR=high(>8) & STAGE=late,OUTCOME=Loss,0.091586,1.0,2.077289,17
3,RRR=high(>8) & WKTS_LEFT=few(<=3),OUTCOME=Loss,0.084144,1.0,2.077289,21
4,PACE_VS_REQ=behind(<-1) & RRR=high(>8) & STAGE...,OUTCOME=Loss,0.082999,1.0,2.077289,17
5,PACE_VS_REQ=behind(<-1) & RRR=high(>8) & WKTS_...,OUTCOME=Loss,0.076703,1.0,2.077289,21
6,RRR=high(>8) & STAGE=middle & WKTS_LEFT=modera...,OUTCOME=Loss,0.065827,1.0,2.077289,14
7,RRR=high(>8) & STAGE=late & WKTS_LEFT=few(<=3),OUTCOME=Loss,0.060675,1.0,2.077289,17
8,PACE_VS_REQ=behind(<-1) & RRR=high(>8) & STAGE...,OUTCOME=Loss,0.060103,1.0,2.077289,14
9,RRR=high(>8) & STAGE=early,OUTCOME=Loss,0.060103,1.0,2.077289,12


## Phases 5-6: grouped CV (quick demo: 1 repeat, LogReg and RandomForest, sets A and B)
The reported results use 5 repeats and all models (`run_experiments.py`).

In [14]:
from models import *
mm, oof, part, tp = run_cv(df, ['LogReg','RandomForest'], {k: FEATURE_SETS[k] for k in ['A_basic','B_basic+momentum']}, n_repeats=1, verbose=False)
mm[['feature_set','model','accuracy','f1','roc_auc','pr_auc','brier']].round(3)

,feature_set,model,accuracy,f1,roc_auc,pr_auc,brier
0,A_basic,LogReg,0.902,0.905,0.962,0.971,0.076
1,A_basic,RandomForest,0.912,0.913,0.946,0.963,0.075
2,B_basic+momentum,LogReg,0.896,0.899,0.961,0.970,0.077
3,B_basic+momentum,RandomForest,0.910,0.911,0.946,0.962,0.076


## Saved full results

In [15]:
pd.read_csv('../results/cv_metrics_summary.csv')[['feature_set','model','accuracy_mean','f1_mean','roc_auc_mean','pr_auc_mean','brier_mean']].round(3)

,feature_set,model,accuracy_mean,f1_mean,roc_auc_mean,pr_auc_mean,brier_mean
0,A_basic,DecisionTree,0.857,0.859,0.875,0.855,0.126
1,A_basic,Dummy(always-win),0.517,0.681,0.500,0.517,0.483
2,A_basic,Dummy(majority),0.389,0.457,0.500,0.517,0.264
3,A_basic,LogReg,0.908,0.911,0.962,0.971,0.073
4,A_basic,NaiveBayes,0.878,0.884,0.956,0.959,0.096
5,A_basic,RandomForest,0.901,0.902,0.946,0.957,0.079
6,B_basic+momentum,DecisionTree,0.857,0.860,0.879,0.853,0.125
7,B_basic+momentum,Dummy(always-win),0.517,0.681,0.500,0.517,0.483
8,B_basic+momentum,Dummy(majority),0.389,0.457,0.500,0.517,0.264
9,B_basic+momentum,LogReg,0.900,0.902,0.961,0.969,0.076


In [16]:
pd.read_csv('../results/ablation_paired_bootstrap.csv').round(4)

,model,new,old,d_auc,d_auc_lo,d_auc_hi,d_brier,d_brier_lo,d_brier_hi
0,LogReg,B_basic+momentum,A_basic,-0.0011,-0.0029,0.0008,0.0018,-0.0022,0.0049
1,LogReg,C_full(+context),B_basic+momentum,0.0078,-0.0339,0.0534,0.0047,-0.0504,0.0571
2,LogReg,C_full(+context),A_basic,0.0067,-0.0348,0.0528,0.0065,-0.0468,0.0585
3,LogReg,D_basic+context,A_basic,-0.0001,-0.0493,0.0476,0.0143,-0.0427,0.0731
4,NaiveBayes,B_basic+momentum,A_basic,-0.0063,-0.0197,0.0132,0.0039,-0.0250,0.0277
5,NaiveBayes,C_full(+context),B_basic+momentum,-0.2136,-0.3573,-0.0731,0.1741,0.0554,0.2943
6,NaiveBayes,C_full(+context),A_basic,-0.2199,-0.3703,-0.0807,0.1780,0.0588,0.2944
7,NaiveBayes,D_basic+context,A_basic,-0.2253,-0.3746,-0.0830,0.1898,0.0667,0.3064
8,DecisionTree,B_basic+momentum,A_basic,-0.0094,-0.0247,-0.0007,0.0058,0.0001,0.0176
9,DecisionTree,C_full(+context),B_basic+momentum,-0.0226,-0.1533,0.1257,0.0336,-0.0709,0.1331


In [17]:
pd.read_csv('../results/checkpoint_metrics.csv').query("feature_set=='B_basic+momentum'")[['model','over','n_matches','accuracy','roc_auc','brier']].round(3)

,model,over,n_matches,accuracy,roc_auc,brier
20,LogReg,10,47,0.894,0.944,0.088
21,LogReg,20,46,0.870,0.947,0.099
22,LogReg,30,41,0.902,0.983,0.060
23,LogReg,40,30,0.933,0.996,0.031
24,LogReg,45,16,1.000,1.000,0.010
25,NaiveBayes,10,47,0.872,0.918,0.117
26,NaiveBayes,20,46,0.891,0.941,0.107
27,NaiveBayes,30,41,1.000,1.000,0.011
28,NaiveBayes,40,30,1.000,1.000,0.003
29,NaiveBayes,45,16,0.938,1.000,0.064


## Figures
See `../figures/`. Example: probability trajectories of all chases (out-of-fold).

In [18]:
img = plt.imread('../figures/fig12a_probability_trajectories_all.png'); print(img.shape)

(494, 1430, 4)
